# 06 - Deployment Guide: End-to-End Setup

This notebook provides step-by-step instructions to deploy the entire RAG application.

## Architecture Diagram

```
┌─────────────────────────────────────────────────────────────────┐
│                    DATA INGESTION LAYER                        │
│  UC Volume (PDFs) → Lakeflow SDP Pipeline                      │
│  [Bronze: Raw Binary] → [Silver: Parsed] → [Gold: Chunks]      │
│                                                                 │
│  Trigger: File Arrival on Volume                                │
└─────────────────────────────────────────────────────────────────┘
                              │
                              ▼
┌─────────────────────────────────────────────────────────────────┐
│                    VECTOR SEARCH LAYER                          │
│  Delta Sync Index (Triggered)                                   │
│  Embedding Model: system.ai.qwen3-embedding-0-6b (1024 dim)    │
│  Source: gold_validated_chunks → Auto-sync on pipeline run      │
└─────────────────────────────────────────────────────────────────┘
                              │
                              ▼
┌─────────────────────────────────────────────────────────────────┐
│                    RAG SERVING LAYER                            │
│  Model Serving Endpoint: rag_project_docs_endpoint              │
│  Chain: Vector Search Retriever → Claude Haiku 4.5             │
│  Features: MLflow Tracing, Inference Logging, Auto-scaling     │
└─────────────────────────────────────────────────────────────────┘
                              │
                              ▼
┌─────────────────────────────────────────────────────────────────┐
│                    APPLICATION LAYER                            │
│  Streamlit Chat App on Databricks Apps                          │
│  Features: Chat history, Source attribution, Error handling     │
└─────────────────────────────────────────────────────────────────┘
```

## Execution Order

| Step | Notebook/Action | Purpose |
|------|----------------|--------|
| 1 | `01_data_setup` | Create UC catalog, schema, volume; copy PDFs |
| 2 | Create Lakeflow Pipeline | Attach `02_ingestion_pipeline` notebook |
| 3 | Run Pipeline | Process PDFs into chunks |
| 4 | `03_vector_search_setup` | Create VS endpoint and Delta Sync index |
| 5 | `04_rag_chain` | Build, test, and deploy RAG endpoint |
| 6 | `05_evaluation` | Evaluate RAG quality |
| 7 | Deploy Streamlit App | Deploy chat UI on Databricks Apps |

## Step 1: Data Infrastructure Setup

Run notebook `01_data_setup` to create:
- Unity Catalog: `ashish_kamboj_rag.project_docs_rag`
- Volume: `/Volumes/ashish_kamboj_rag/project_docs_rag/raw_documents/`
- Copies 15 PDFs from workspace to the Volume

> **Action:** Run `01_data_setup` interactively on any cluster.

## Step 2: Create Lakeflow Spark Declarative Pipeline

### Pipeline Configuration

Create a new pipeline with these settings:

| Setting | Value |
|---------|-------|
| **Name** | `rag_document_ingestion` |
| **Source Code** | `/Users/ashish.kamboj.contractor@pepsico.com/rag_chatbot/02_ingestion_pipeline` |
| **Target Catalog** | `ashish_kamboj_rag` |
| **Target Schema** | `project_docs_rag` |
| **Pipeline Mode** | Triggered |
| **Channel** | Preview (required for `ai_parse_document` and `ai_prep_search`) |

### File Arrival Trigger (Automation)

To automatically run the pipeline when new files are uploaded:

1. **Create a Lakeflow Job** with the pipeline as a task
2. **Add a File Arrival Trigger**:
   - Trigger type: `File arrival`
   - Volume path: `/Volumes/ashish_kamboj_rag/project_docs_rag/raw_documents/`
   - Min time between triggers: 5 minutes (prevents rapid re-runs)

> **Important Note on Trigger Feasibility:**
> - ✅ **UC Volume path** supports file arrival triggers natively
> - ❌ **Workspace folder** (`/Users/.../Project_Docs/`) does NOT support file arrival triggers
> - This is why we copy files from the workspace to a Volume in Step 1
> - For ongoing use, upload new documents directly to the Volume

### Alternative: Scheduled Trigger
If file arrival triggers are not available in your workspace, use a **scheduled trigger** (e.g., hourly) on the Lakeflow Job instead.

In [0]:
# =============================================================================
# CREATE LAKEFLOW PIPELINE PROGRAMMATICALLY
# =============================================================================
# This cell creates the pipeline using the Databricks SDK.
# Alternatively, create it via the UI following the settings above.
# =============================================================================

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.pipelines import (
    PipelineLibrary,
    NotebookLibrary
)

w = WorkspaceClient()

# Define the pipeline with serverless compute (required in this workspace)
pipeline = w.pipelines.create(
    name="rag_document_ingestion",
    catalog="ashish_kamboj_rag",
    target="project_docs_rag",
    channel="PREVIEW",  # Required for ai_parse_document and ai_prep_search
    serverless=True,    # Use serverless compute
    libraries=[
        PipelineLibrary(
            notebook=NotebookLibrary(
                path="/Users/ashu.009kamboj@gmail.com/rag_chatbot/02_ingestion_pipeline.sql"
            )
        )
    ],
    continuous=False,  # Triggered mode
    development=False  # Production mode
)

print(f"✅ Pipeline created: {pipeline.pipeline_id}")
print(f"   Name: rag_document_ingestion")
print(f"   Navigate to Pipelines UI to view and run.")

✅ Pipeline created: 49d4d5bb-94c5-4ff1-9837-d94695bad56b
   Name: rag_document_ingestion
   Navigate to Pipelines UI to view and run.


In [0]:
# =============================================================================
# CREATE JOB WITH FILE ARRIVAL TRIGGER
# =============================================================================
# Sets up a Job that runs the pipeline when new files arrive in the Volume.
# =============================================================================

from databricks.sdk.service.jobs import (
    Task,
    PipelineTask,
    FileArrivalTriggerConfiguration,
    TriggerSettings,
    PauseStatus
)

# Create the job with file arrival trigger
job = w.jobs.create(
    name="rag_ingestion_file_trigger",
    tasks=[
        Task(
            task_key="run_ingestion_pipeline",
            pipeline_task=PipelineTask(
                pipeline_id=pipeline.pipeline_id,
                full_refresh=False  # Incremental processing
            )
        )
    ],
    trigger=TriggerSettings(
        file_arrival=FileArrivalTriggerConfiguration(
            url="/Volumes/ashish_kamboj_rag/project_docs_rag/raw_documents/",
            min_time_between_triggers_seconds=300,  # 5 min minimum between runs
            wait_after_last_change_seconds=60  # Wait 60s after last file change
        ),
        pause_status=PauseStatus.UNPAUSED
    )
)

print(f"✅ Job created with file arrival trigger: {job.job_id}")
print(f"   Trigger: New files in /Volumes/ashish_kamboj_rag/project_docs_rag/raw_documents/")
print(f"   Min interval: 5 minutes")
print(f"   The pipeline will automatically run when new PDFs are uploaded to the Volume.")

✅ Job created with file arrival trigger: 896339934817140
   Trigger: New files in /Volumes/ashish_kamboj_rag/project_docs_rag/raw_documents/
   Min interval: 5 minutes
   The pipeline will automatically run when new PDFs are uploaded to the Volume.


## Step 3: Run the Pipeline

After creating the pipeline, trigger an initial run:
- Go to the Pipelines UI and click "Start"
- Or the file arrival trigger will run it when files land in the Volume
- Monitor progress in the Pipeline UI (Bronze → Silver → Gold)

## Step 4: Vector Search Setup

Run `03_vector_search_setup` to:
- Create the Vector Search endpoint
- Create the Delta Sync Index pointing to `gold_validated_chunks`
- Test with a sample similarity search

> **Note:** Index creation takes 5-15 minutes. The notebook waits for completion.

## Step 5: RAG Chain Deployment

Run `04_rag_chain` to:
- Build the LangChain RAG chain
- Test with sample questions
- Log to MLflow with model signature
- Deploy to a Model Serving endpoint

> **Note:** Endpoint provisioning takes 5-10 minutes.

## Step 6: Deploy Streamlit App on Databricks Apps

### Deployment Steps:

1. **Navigate to Databricks Apps** in the left sidebar
2. **Create New App:**
   - Name: `rag-project-docs`
   - Source: `/Users/ashish.kamboj.contractor@pepsico.com/rag_chatbot/streamlit_app/`
3. **The app deployment reads:**
   - `app.py` - Main Streamlit application
   - `app.yaml` - Configuration (command, env vars, resources)
   - `requirements.txt` - Python dependencies

### What `app.yaml` Configures:
- **Command:** Runs Streamlit on the correct port (`$DATABRICKS_APP_PORT`)
- **Environment:** Sets `SERVING_ENDPOINT` to `rag_project_docs_endpoint`
- **Resources:** Grants the app's service principal `CAN_QUERY` permission on the RAG endpoint

### Authentication:
Databricks Apps automatically provides:
- `DATABRICKS_HOST` - Workspace URL
- `DATABRICKS_TOKEN` - OAuth token for the app's service principal

No manual token management required.

## Step 7: Evaluation & Ongoing Monitoring

### Initial Evaluation
Run `05_evaluation` to:
- Score the RAG chain across multiple dimensions (Relevance, Groundedness, Safety)
- View results in the MLflow Experiment UI
- Identify weak areas for improvement

### Ongoing Monitoring

**MLflow Tracing** is automatically enabled:
- Every query to the RAG endpoint generates a trace
- Traces capture: input query, retrieved chunks, LLM prompt, generated response, latency
- View traces in the MLflow Experiment UI

**Inference Logging** (enabled in endpoint config):
- All requests/responses are logged to a Delta table
- Table: `ashish_kamboj_rag.project_docs_rag.__endpoint_inference_log`
- Use for drift detection, usage analytics, and quality monitoring

### Improving Quality

| Issue | Solution |
|-------|----------|
| Irrelevant retrievals | Adjust `NUM_RESULTS`, try different embedding models |
| Incomplete answers | Increase `MAX_TOKENS`, improve chunking parameters |
| Hallucinations | Lower `TEMPERATURE`, strengthen system prompt |
| Missing info | Add more source documents to the Volume |
| Stale answers | Pipeline auto-runs on new files via trigger |

## Appendix: File-Based Trigger Feasibility

### ✅ Supported: UC Volume Path
The file arrival trigger works natively with Unity Catalog Volumes:
- Path: `/Volumes/ashish_kamboj_rag/project_docs_rag/raw_documents/`
- Mechanism: Databricks monitors the Volume for new/modified files
- Latency: Files detected within 1-5 minutes
- Supported operations: File upload, file delete (triggers on changes)

### ❌ Not Supported: Workspace Folder
The original path (`/Users/.../Project_Docs/`) is a **workspace folder**, not cloud object storage.
- Workspace folders do NOT support file arrival triggers
- Workspace folders are not optimized for Auto Loader
- **Workaround:** Copy files to a UC Volume (done in `01_data_setup`)

### Recommendation for Production
For ongoing document management:
1. Upload new PDFs directly to the Volume: `/Volumes/ashish_kamboj_rag/project_docs_rag/raw_documents/`
2. The file arrival trigger detects the new file
3. The pipeline runs automatically (incremental - only processes new files)
4. Vector Search index syncs with the updated chunks table
5. RAG endpoint immediately serves answers from the new content

### Handling File Deletions
- When a file is deleted from the Volume, the pipeline re-runs
- However, Auto Loader processes files incrementally (append-only)
- For true delete propagation, you would need to:
  1. Drop and recreate the streaming tables (full refresh)
  2. Or implement a separate cleanup job that removes chunks from deleted files